# AIPW analytic M-estimation simulation

This notebook checks that `aipw_m_estimation_variance` reproduces the existing AIPW covariance estimator, then compares empirical Monte Carlo variance with the analytic sandwich variance.

Estimand: the lower-triangular entries of the full-data covariance matrix under the Gaussian two-factor DGP. The run uses MAR conditional on `Z`, the correctly specified linear conditional-mean model, the shared binomial-logit response model, 1,000 replications, and seed `20260922`.

In [5]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'aipw.py').exists():
    ROOT = ROOT / 'python'
sys.path.insert(0, str(ROOT))

from aipw import aipw_z
from aipw_m_estimation import aipw_m_estimation_variance
from dgp import DGPTwoFac

def fit_current_propensity(response, z):
    observed = response.sum(axis=1)
    missing = response.shape[1] - observed
    model = sm.GLM(
        np.column_stack([observed, missing]),
        sm.add_constant(z),
        family=sm.families.Binomial(),
    ).fit(disp=0)
    return model.params, model.predict()

def lower_triangle(matrix):
    rows, cols = np.tril_indices(matrix.shape[0])
    return matrix[rows, cols]

np.set_printoptions(precision=6, suppress=True, linewidth=160, threshold=np.inf)

In [ ]:
N = 500
REPETITIONS = 1000
SEED = 20260922
DGP_KW = dict(p=6, loading=0.8, phi_offdiag=0.3, cov_z_x=0.64, mu=2.0, pi_offset=1.4)
print({'n': N, 'repetitions': REPETITIONS, 'seed': SEED, **DGP_KW})

In [ ]:
dgp = DGPTwoFac(N, **DGP_KW)
dgp.gen(seed=SEED)
response = (~np.isnan(dgp.X_o)).astype(float)
alpha_hat, pi_hat = fit_current_propensity(response, dgp.Z)
m_est = aipw_m_estimation_variance(dgp.X_o, dgp.Z, alpha=alpha_hat)
legacy = aipw_z(dgp.X_o, dgp.Z, np.repeat(pi_hat[:, None], dgp.p, axis=1))

print('Maximum legacy/new point-estimate difference:', np.max(np.abs(legacy - m_est['covariance'])))
print('Maximum absolute stacked score mean:', np.max(np.abs(m_est['score_mean'])))
print('Bread condition number:', m_est['bread_condition_number'])
print('\nalpha_hat:')
print(m_est['alpha'])
print('\nbeta_hat:')
print(m_est['beta'])
print('\npsi_bar:')
print(m_est['score_mean'])
print('\npsi for the first observation:')
print(m_est['scores'][0])
print('\nAnalytic Jacobian / bread:')
print(m_est['bread'])
print('\nMeat:')
print(m_est['meat'])
print('\nSandwich covariance for vech(covariance):')
print(m_est['covariance_covariance'])

Maximum legacy/new point-estimate difference: 6.217248937900877e-15
Maximum absolute stacked score mean: 1.0103495817759267e-13
Bread condition number: 71.16960749874131

alpha_hat:
[-0.613085  1.007723]

beta_hat:
[[0.670318 0.680764 0.776903 0.732567 0.850348 0.917604]
 [0.668566 0.66597  0.609531 0.607242 0.585125 0.571436]]

psi_bar:
[ 0.  0. -0. -0. -0. -0.  0.  0.  0.  0. -0. -0.  0.  0.  0.  0. -0. -0.  0. -0.  0.  0. -0. -0. -0. -0. -0. -0. -0.  0.  0. -0.  0. -0. -0. -0. -0. -0.  0.
  0.  0.  0. -0.  0. -0. -0. -0. -0.  0.  0.  0. -0. -0.  0. -0. -0. -0. -0. -0.  0.  0.  0.  0. -0.  0.  0. -0.  0.  0.  0. -0.  0.  0. -0.  0.  0.  0. -0.
  0. -0.  0. -0.  0. -0. -0. -0.  0.  0.  0.  0.  0. -0.  0. -0. -0. -0.  0.  0.]

psi for the first observation:
[ 0.901764  3.938112  1.389935  6.070016  0.296573  1.295169 -0.896654 -3.915799 -1.995053 -8.712638 -1.764519 -7.705869  0.132868  0.580252  0.901764
  0.88219   0.933964  0.881122  0.912353  0.892765  1.407803  1.389935  1.392097 

In [3]:
dgp = DGPTwoFac(N, **DGP_KW)
dgp.gen()
dgp._rng = np.random.default_rng(SEED)
pairs = list(zip(*np.tril_indices(dgp.p)))
truth = lower_triangle(dgp.Lambda @ dgp.Phi @ dgp.Lambda.T + dgp.Omega)
legacy_estimates = []
m_estimates = []
sandwich_variances = []
point_differences = []

for replication in range(REPETITIONS):
    dgp.gen()
    response = (~np.isnan(dgp.X_o)).astype(float)
    alpha_hat, pi_hat = fit_current_propensity(response, dgp.Z)
    m_est = aipw_m_estimation_variance(dgp.X_o, dgp.Z, alpha=alpha_hat)
    legacy = aipw_z(dgp.X_o, dgp.Z, np.repeat(pi_hat[:, None], dgp.p, axis=1))
    legacy_vech = lower_triangle(legacy)
    m_vech = lower_triangle(m_est['covariance'])
    legacy_estimates.append(legacy_vech)
    m_estimates.append(m_vech)
    sandwich_variances.append(np.diag(m_est['covariance_covariance']))
    point_differences.append(np.max(np.abs(legacy_vech - m_vech)))

legacy_estimates = np.asarray(legacy_estimates)
m_estimates = np.asarray(m_estimates)
sandwich_variances = np.asarray(sandwich_variances)

summary = pd.DataFrame({
    'pair': [f'({j + 1}, {k + 1})' for j, k in pairs],
    'truth': truth,
    'legacy_bias': legacy_estimates.mean(axis=0) - truth,
    'm_estimation_bias': m_estimates.mean(axis=0) - truth,
    'empirical_variance': m_estimates.var(axis=0, ddof=1),
    'mean_sandwich_variance': sandwich_variances.mean(axis=0),
})
summary['variance_ratio'] = summary['mean_sandwich_variance'] / summary['empirical_variance']

print('Maximum legacy/new point-estimate difference over all replications:', max(point_differences))
print('Maximum legacy/new bias difference:', np.max(np.abs(summary['legacy_bias'] - summary['m_estimation_bias'])))
display(summary)
print('\nTotal empirical variance:', summary['empirical_variance'].sum())
print('Total mean sandwich variance:', summary['mean_sandwich_variance'].sum())
print('Mean variance ratio:', summary['variance_ratio'].mean())

Maximum legacy/new point-estimate difference over all replications: 1.5987211554602254e-14
Maximum legacy/new bias difference: 0.0
pair  truth  legacy_bias  m_estimation_bias  empirical_variance  mean_sandwich_variance  variance_ratio
(1, 1)  1.000000  -0.003060  -0.003060  0.005602  0.005935  1.059325
(2, 1)  0.640000  0.003760  0.003760  0.007832  0.007687  0.981387
(2, 2)  1.000000  0.001397  0.001397  0.006156  0.006113  0.993042
(3, 1)  0.640000  -0.004263  -0.004263  0.007911  0.007569  0.956753
(3, 2)  0.640000  -0.000878  -0.000878  0.008021  0.007729  0.963523
(3, 3)  1.000000  -0.004326  -0.004326  0.006601  0.006008  0.910161
(4, 1)  0.192000  -0.001548  -0.001548  0.006926  0.006864  0.991010
(4, 2)  0.192000  -0.002504  -0.002504  0.006868  0.006836  0.995435
(4, 3)  0.192000  -0.002696  -0.002696  0.006882  0.006759  0.982080
(4, 4)  1.000000  -0.007163  -0.007163  0.005736  0.005950  1.037202
(5, 1)  0.192000  -0.001049  -0.001049  0.007163  0.006891  0.961988
(5, 2)  0.

In [6]:
N = 1000
REPETITIONS = 100
SEED = 293842938

dgp = DGPTwoFac(N, **DGP_KW)
dgp.gen()
dgp._rng = np.random.default_rng(SEED)
pairs = list(zip(*np.tril_indices(dgp.p)))
truth = lower_triangle(dgp.Lambda @ dgp.Phi @ dgp.Lambda.T + dgp.Omega)
legacy_estimates = []
m_estimates = []
sandwich_variances = []
point_differences = []

for replication in range(REPETITIONS):
    dgp.gen()
    response = (~np.isnan(dgp.X_o)).astype(float)
    alpha_hat, pi_hat = fit_current_propensity(response, dgp.Z)
    m_est = aipw_m_estimation_variance(dgp.X_o, dgp.Z, alpha=alpha_hat)
    legacy = aipw_z(dgp.X_o, dgp.Z, np.repeat(pi_hat[:, None], dgp.p, axis=1))
    legacy_vech = lower_triangle(legacy)
    m_vech = lower_triangle(m_est['covariance'])
    legacy_estimates.append(legacy_vech)
    m_estimates.append(m_vech)
    sandwich_variances.append(np.diag(m_est['covariance_covariance']))
    point_differences.append(np.max(np.abs(legacy_vech - m_vech)))

legacy_estimates = np.asarray(legacy_estimates)
m_estimates = np.asarray(m_estimates)
sandwich_variances = np.asarray(sandwich_variances)

summary = pd.DataFrame({
    'pair': [f'({j + 1}, {k + 1})' for j, k in pairs],
    'truth': truth,
    'legacy_bias': legacy_estimates.mean(axis=0) - truth,
    'm_estimation_bias': m_estimates.mean(axis=0) - truth,
    'empirical_variance': m_estimates.var(axis=0, ddof=1),
    'mean_sandwich_variance': sandwich_variances.mean(axis=0),
})
summary['variance_ratio'] = summary['mean_sandwich_variance'] / summary['empirical_variance']

print('Maximum legacy/new point-estimate difference over all replications:', max(point_differences))
print('Maximum legacy/new bias difference:', np.max(np.abs(summary['legacy_bias'] - summary['m_estimation_bias'])))
display(summary)
print('\nTotal empirical variance:', summary['empirical_variance'].sum())
print('Total mean sandwich variance:', summary['mean_sandwich_variance'].sum())
print('Mean variance ratio:', summary['variance_ratio'].mean())

Maximum legacy/new point-estimate difference over all replications: 1.7319479184152442e-14
Maximum legacy/new bias difference: 1.2212453270876722e-15


,pair,truth,legacy_bias,m_estimation_bias,empirical_variance,mean_sandwich_variance,variance_ratio
0,"(1, 1)",1.000,-0.006207,-0.006207,0.003963,0.003033,0.765531
1,"(2, 1)",0.640,0.007129,0.007129,0.004768,0.003874,0.812568
2,"(2, 2)",1.000,-0.005874,-0.005874,0.003308,0.002939,0.888464
3,"(3, 1)",0.640,-0.010806,-0.010806,0.003510,0.003818,1.087839
4,"(3, 2)",0.640,-0.003346,-0.003346,0.004957,0.004282,0.863926
5,"(3, 3)",1.000,-0.007720,-0.007720,0.002709,0.003105,1.146109
6,"(4, 1)",0.192,-0.003393,-0.003393,0.004257,0.003451,0.810717
7,"(4, 2)",0.192,-0.002931,-0.002931,0.004153,0.003584,0.862950
8,"(4, 3)",0.192,-0.010861,-0.010861,0.003497,0.003654,1.044933
9,"(4, 4)",1.000,-0.002723,-0.002723,0.003274,0.003167,0.967144



Total empirical variance: 0.0758211768788012
Total mean sandwich variance: 0.07460189955136066
Mean variance ratio: 1.005237325937882


In [12]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

ROOT = Path.cwd()
if not (ROOT / "aipw_m_estimation.py").exists():
    ROOT = ROOT / "python"

sys.path.insert(0, str(ROOT))

from dgp import DGPTwoFac
from aipw_m_estimation import aipw_m_estimation_variance

# Generate one reproducible MAR dataset
n = 500
seed = 20260922

dgp = DGPTwoFac(
    n=n,
    p=6,
    loading=0.8,
    phi_offdiag=0.3,
    cov_z_x=0.64,
    mu=2.0,
    pi_offset=1.4,
)
dgp.gen(seed=seed)

Y = dgp.X_o
Z = dgp.Z
truth = dgp.sig_X

# Fit the shared binomial-logit response model
R = (~np.isnan(Y)).astype(float)
observed_count = R.sum(axis=1)
missing_count = Y.shape[1] - observed_count

propensity_fit = sm.GLM(
    np.column_stack([observed_count, missing_count]),
    sm.add_constant(Z),
    family=sm.families.Binomial(),
).fit(disp=0)

# Estimate AIPW covariance and analytic M-estimation sandwich variance
result = aipw_m_estimation_variance(
    Y,
    Z_outcome=Z,
    Z_propensity=Z,
    alpha=propensity_fit.params,
)

p = Y.shape[1]
rows, cols = np.tril_indices(p)
pairs = list(zip(rows, cols))

m_estimation_variance_matrix = np.full((p, p), np.nan)
m_estimation_variance_matrix[rows, cols] = np.diag(
    result["covariance_covariance"]
)
m_estimation_variance_matrix[cols, rows] = np.diag(
    result["covariance_covariance"]
)

variance_table = pd.DataFrame({
    "pair": [f"Cov(X{j + 1}, X{k + 1})" for j, k in pairs],
    "truth": truth[rows, cols],
    "aipw_covariance": result["covariance"][rows, cols],
    "m_estimation_variance": np.diag(result["covariance_covariance"]),
    "m_estimation_se": np.sqrt(np.diag(result["covariance_covariance"])),
})

average_m_estimation_variance = np.mean(
    np.diag(result["covariance_covariance"])
)

print("AIPW covariance matrix:")
print(result["covariance"])

print("\nM-estimation variance for each covariance entry:")
print(m_estimation_variance_matrix)

print(
    "\nAverage marginal M-estimation variance:",
    average_m_estimation_variance,
)

print(
    "Maximum absolute mean score:",
    np.max(np.abs(result["score_mean"])),
)

display(variance_table)

AIPW covariance matrix:
[[0.96423  0.699159 0.532876 0.206213 0.198297 0.149145]
 [0.699159 1.107194 0.601486 0.122566 0.059596 0.077294]
 [0.532876 0.601486 0.917558 0.192077 0.109243 0.191156]
 [0.206213 0.122566 0.192077 0.936984 0.475406 0.662355]
 [0.198297 0.059596 0.109243 0.475406 1.011577 0.513395]
 [0.149145 0.077294 0.191156 0.662355 0.513395 0.855306]]

M-estimation variance for each covariance entry:
[[0.00497  0.007846 0.00618  0.005732 0.007718 0.00585 ]
 [0.007846 0.007535 0.007344 0.006687 0.00828  0.007239]
 [0.00618  0.007344 0.004376 0.006071 0.006545 0.005972]
 [0.005732 0.006687 0.006071 0.006263 0.007911 0.007119]
 [0.007718 0.00828  0.006545 0.007911 0.005055 0.008432]
 [0.00585  0.007239 0.005972 0.007119 0.008432 0.004258]]

Average marginal M-estimation variance: 0.0065420451756074095
Maximum absolute mean score: 1.0103495817759267e-13


,pair,truth,aipw_covariance,m_estimation_variance,m_estimation_se
0,"Cov(X1, X1)",1.000,0.964230,0.004970,0.070497
1,"Cov(X2, X1)",0.640,0.699159,0.007846,0.088578
2,"Cov(X2, X2)",1.000,1.107194,0.007535,0.086802
3,"Cov(X3, X1)",0.640,0.532876,0.006180,0.078611
4,"Cov(X3, X2)",0.640,0.601486,0.007344,0.085700
5,"Cov(X3, X3)",1.000,0.917558,0.004376,0.066154
6,"Cov(X4, X1)",0.192,0.206213,0.005732,0.075709
7,"Cov(X4, X2)",0.192,0.122566,0.006687,0.081776
8,"Cov(X4, X3)",0.192,0.192077,0.006071,0.077914
9,"Cov(X4, X4)",1.000,0.936984,0.006263,0.079140


## Joint AIPW and CFA delta-method variance calibration

This run uses the same two-factor MAR DGP as above. The default report follows the notation in the derivation: beta, alpha, gamma = (mu, nu), and the factor parameters (Lambda, Phi, Omega). The rho and sigma coordinates are retained only as optional implementation detail because they are needed for the current exact sandwich calculation. The simulation loop uses one prevalidated numerical-Jacobian step for speed; set `RUN_DERIVATIVE_VALIDATION = True` to run the slower three-step and directional-check pilot before a production run.

In [ ]:
import os

from joblib import Parallel, delayed, parallel_config
from tqdm.auto import tqdm

from cfa_delta import aipw_and_cfa_m_estimation_variance

# Keep these aligned with the primary AIPW simulation above.
CFA_N = 500
CFA_REPETITIONS = 1000
CFA_SEED = 20260922
SHOW_AUXILIARY_DETAIL = False
SHOW_REPORTED_COVARIANCE_MATRICES = False

# Production simulation uses the step size validated in the pilot below.
# This shortcut affects only the simulation loop, never the one-sample
# M-estimation variance function.
SIMULATION_STEP_SCALES = (1e-5,)
SIMULATION_DIRECTION_CHECKS = 0
RUN_DERIVATIVE_VALIDATION = False
SIMULATION_WORKERS = os.cpu_count() or 1

if RUN_DERIVATIVE_VALIDATION:
    pilot_dgp = DGPTwoFac(CFA_N, **DGP_KW)
    pilot_dgp.gen(seed=CFA_SEED + 1)
    pilot_response = (~np.isnan(pilot_dgp.X_o)).astype(float)
    pilot_alpha, _ = fit_current_propensity(pilot_response, pilot_dgp.Z)
    pilot_aipw = aipw_m_estimation_variance(
        pilot_dgp.X_o, pilot_dgp.Z, alpha=pilot_alpha
    )
    pilot_joint = aipw_and_cfa_m_estimation_variance(
        pilot_aipw, n_obs=CFA_N
    )
    print('Full derivative-validation pilot:')
    print('Selected step:', pilot_joint['cfa_result']['selected_step_scale'])
    display(pd.DataFrame(pilot_joint['cfa_result']['stability']).T)
    display(pd.DataFrame(pilot_joint['cfa_result']['direction_check']))

def run_simulation_replication(replication, seed):
    dgp = DGPTwoFac(CFA_N, **DGP_KW)
    dgp.gen(seed=int(seed))
    response = (~np.isnan(dgp.X_o)).astype(float)
    alpha_hat, _ = fit_current_propensity(response, dgp.Z)
    aipw_result = aipw_m_estimation_variance(
        dgp.X_o, dgp.Z, alpha=alpha_hat
    )
    try:
        joint_result = aipw_and_cfa_m_estimation_variance(
            aipw_result,
            n_obs=CFA_N,
            step_scales=SIMULATION_STEP_SCALES,
            n_direction_checks=SIMULATION_DIRECTION_CHECKS,
        )
    except (RuntimeError, ValueError) as error:
        return {'replication': replication, 'error': str(error)}
    return {
        'replication': replication,
        'parameters': joint_result['parameter_values'],
        'covariance': joint_result['joint_covariance'],
        'parameter_names': joint_result['parameter_names'],
        'parameter_groups': joint_result['parameter_groups'],
        'score_mean': aipw_result['score_mean'],
        'parameter_slices': aipw_result['parameter_slices'],
    }

seed_sequence = np.random.SeedSequence(CFA_SEED)
replication_seeds = seed_sequence.generate_state(CFA_REPETITIONS, dtype=np.uint32)
simulation_workers = min(SIMULATION_WORKERS, CFA_REPETITIONS)
print(f'Running {CFA_REPETITIONS} replications with {simulation_workers} worker processes.')

simulation_results = []
with parallel_config(backend='loky', inner_max_num_threads=1):
    with Parallel(
        n_jobs=simulation_workers,
        return_as='generator_unordered',
        pre_dispatch='2*n_jobs',
    ) as parallel:
        result_stream = parallel(
            delayed(run_simulation_replication)(replication, seed)
            for replication, seed in enumerate(replication_seeds)
        )
        for result in tqdm(result_stream, total=CFA_REPETITIONS, desc='CFA simulations'):
            simulation_results.append(result)

simulation_results.sort(key=lambda result: result['replication'])
successful_results = [result for result in simulation_results if 'error' not in result]
failed_replications = [
    (result['replication'], result['error'])
    for result in simulation_results if 'error' in result
]
all_parameters = np.asarray([result['parameters'] for result in successful_results])
predicted_joint_covariances = np.asarray([result['covariance'] for result in successful_results])
if not successful_results:
    raise RuntimeError('No successful replications; inspect failed_replications.')
joint_result = successful_results[-1]
aipw_result = {
    'score_mean': successful_results[-1]['score_mean'],
    'parameter_slices': successful_results[-1]['parameter_slices'],
}

if len(failed_replications):
    print(f'CFA/delta failures: {len(failed_replications)} of {CFA_REPETITIONS}')
    display(pd.DataFrame(failed_replications, columns=['replication', 'reason']).head())
else:
    print(f'All {CFA_REPETITIONS} CFA/delta evaluations completed.')

if all_parameters.shape[0] < 2:
    raise RuntimeError('Fewer than two successful replications; variance cannot be estimated.')

empirical_joint_covariance = np.cov(all_parameters, rowvar=False, ddof=1)
mean_predicted_joint_covariance = predicted_joint_covariances.mean(axis=0)
empirical_parameter_variance = np.diag(empirical_joint_covariance)
mean_predicted_parameter_variance = np.diag(mean_predicted_joint_covariance)

# The scalar total is the trace: the sum of marginal parameter variances.
# It does not include off-diagonal covariances, which are not variances.
# CFA coordinates are derived from the AIPW stack, so this 111-coordinate
# trace is an accounting summary, not an independent-information total.
total_empirical_variance = np.trace(empirical_joint_covariance)
total_predicted_variance = np.trace(mean_predicted_joint_covariance)

cfa_variance_comparison = pd.DataFrame({
    'group': joint_result['parameter_groups'],
    'parameter': joint_result['parameter_names'],
    'empirical_variance': empirical_parameter_variance,
    'mean_delta_variance': mean_predicted_parameter_variance,
})
cfa_variance_comparison['variance_ratio'] = (
    cfa_variance_comparison['mean_delta_variance']
    / cfa_variance_comparison['empirical_variance']
)
cfa_variance_comparison['empirical_variance_contribution'] = (
    cfa_variance_comparison['empirical_variance'] / total_empirical_variance
)
cfa_variance_comparison['predicted_variance_contribution'] = (
    cfa_variance_comparison['mean_delta_variance'] / total_predicted_variance
)
cfa_variance_comparison['variance_error'] = (
    cfa_variance_comparison['mean_delta_variance']
    - cfa_variance_comparison['empirical_variance']
)

# Gamma is the 27-vector (mu, nu) in the derivation. Psi denotes the
# estimating functions, so it is reported below as a zero-mean diagnostic.
cfa_variance_comparison['reported_group'] = cfa_variance_comparison['group'].replace({
    'mu': 'gamma',
    'nu': 'gamma',
    'cfa': 'Lambda_Phi_Omega',
})
reported_groups = ['beta', 'alpha', 'gamma', 'Lambda_Phi_Omega']
reported_mask = cfa_variance_comparison['reported_group'].isin(reported_groups)
reported_indices = np.flatnonzero(reported_mask.to_numpy())
reported_parameter_comparison = cfa_variance_comparison.loc[reported_mask].copy()

group_variance_comparison = reported_parameter_comparison.groupby('reported_group', sort=False)[[
    'empirical_variance', 'mean_delta_variance'
]].sum()
group_variance_comparison['variance_ratio'] = (
    group_variance_comparison['mean_delta_variance']
    / group_variance_comparison['empirical_variance']
)

reported_empirical_covariance = empirical_joint_covariance[np.ix_(reported_indices, reported_indices)]
reported_predicted_covariance = mean_predicted_joint_covariance[np.ix_(reported_indices, reported_indices)]
reported_total_empirical_variance = np.trace(reported_empirical_covariance)
reported_total_predicted_variance = np.trace(reported_predicted_covariance)
reported_parameter_comparison['empirical_variance_contribution'] = (
    reported_parameter_comparison['empirical_variance'] / reported_total_empirical_variance
)
reported_parameter_comparison['predicted_variance_contribution'] = (
    reported_parameter_comparison['mean_delta_variance'] / reported_total_predicted_variance
)

if SHOW_REPORTED_COVARIANCE_MATRICES:
    print('\nEmpirical covariance for beta, alpha, gamma, and Lambda/Phi/Omega:')
    print(reported_empirical_covariance)
    print('\nMean predicted covariance for beta, alpha, gamma, and Lambda/Phi/Omega:')
    print(reported_predicted_covariance)
print('\nTotal empirical variance across the 54 reported coordinates (trace):', reported_total_empirical_variance)
print('Total mean predicted variance across the 54 reported coordinates (trace):', reported_total_predicted_variance)
print('Total variance ratio:', reported_total_predicted_variance / reported_total_empirical_variance)
print('\nVariance by reported parameter group:')
display(group_variance_comparison)
print('\nParameter-wise variance comparison (default reporting blocks):')
display(reported_parameter_comparison)

slices = aipw_result['parameter_slices']
psi_block_check = pd.DataFrame({
    'block': ['beta', 'alpha', 'gamma'],
    'max_absolute_mean_psi': [
        np.max(np.abs(aipw_result['score_mean'][slices.beta])),
        np.max(np.abs(aipw_result['score_mean'][slices.alpha])),
        np.max(np.abs(aipw_result['score_mean'][slices.mu.start:slices.nu.stop])),
    ],
})
print('\nPsi estimating-equation check from the last replication:')
display(psi_block_check)

if SHOW_AUXILIARY_DETAIL:
    print('\nAuxiliary rho and sigma detail, plus the full 111-coordinate covariance:')
    print(empirical_joint_covariance)
    print(mean_predicted_joint_covariance)
    display(cfa_variance_comparison)